# Assignment 3: Experimentation and Expansion
## FAST-NUCES — Department of Artificial Intelligence & Data Science
**Course:** Deep Learning (DS/AI)  
**Instructor:** Mr. Ubaid Ul Rehman  
**Authors:** Yahya Abdul Hakeem (23I-2627) · Mustafa Kashif (23I-2583) · Ishmal Faheem (23I-5032)

---

### Overview
This notebook extends Assignment 2 (RecDCL reproduction) in two directions:

1. **Mandatory — Cross-dataset evaluation:** Run all A2 models (MF, LightGCN, RecDCL) on Amazon Beauty, a new benchmark from the RecDCL paper family, to validate findings across datasets.

2. **Proposed Method — AdaptRecDCL:** We observed in A2 that RecDCL's fixed loss weights (λ_BCL, λ_FCL) cause training collapse at full scale. Our proposed fix replaces fixed λ with **learnable weights automatically balanced via GradNorm** (Chen et al., ICML 2018), eliminating per-dataset manual tuning.

### Models
| Model | Type | Description |
|-------|------|-------------|
| MF | Baseline | Matrix Factorization with BPR loss |
| LightGCN | Graph CF | Simplified GCN for recommendation |
| RecDCL | SSL (A2 paper) | Dual Contrastive Learning — fixed λ |
| **AdaptRecDCL** | **Proposed** | RecDCL with GradNorm-adaptive λ weighting |

### Datasets
| Dataset | Users | Items | Density | Source |
|---------|-------|-------|---------|--------|
| Gowalla (subset) | 3,000 | ~33K | 0.08% | SNAP |
| Amazon Beauty | ~35K | ~18K | ~0.06% | McAuley et al. |

### Expected Runtime (Colab T4 GPU)
- Gowalla: ~10 minutes
- Amazon Beauty: ~45 minutes
- **Total: ~55 minutes**


---
## Step 1: Install Dependencies


In [1]:
!pip install torch matplotlib requests numpy -q
print("✓ Dependencies ready")


✓ Dependencies ready


---
## Step 2: Imports and Device Configuration
We check for GPU availability. A T4 GPU on Colab is recommended.  
If no GPU is found, the script falls back to CPU (much slower).


In [2]:
import os, json, math, time, random, gzip
import numpy as np
import requests
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Device ────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Runtime will be significantly longer.")
    print("Go to Runtime → Change runtime type → T4 GPU")

# ── Output directory ──────────────────────────────────────────
os.makedirs("results_a3", exist_ok=True)
print("\nOutput directory: results_a3/")

# ── Seeding ───────────────────────────────────────────────────
SEED = 2024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(f"Seed   : {SEED}")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

Output directory: results_a3/
Seed   : 2024


---
## Step 3: Hyperparameter Configuration

All hyperparameters are defined here in one place for reproducibility.  
Key settings match the RecDCL paper where applicable.

**AdaptRecDCL-specific parameters:**
- `adapt_alpha`: GradNorm restoring force — controls how aggressively λ values are rebalanced
- `adapt_lr`: Learning rate for the λ parameters (separate from main model LR)


In [3]:
CFG = {
    "emb_size"   : 64,      # embedding dimension (matches paper)
    "gnn_layers" : 3,       # LightGCN propagation layers (matches paper)
    "batch_size" : 2048,    # matches paper
    "topk"       : 20,      # Recall@20, NDCG@20
    "l2_reg"     : 1e-4,    # L2 regularisation weight
    "bcl_temp"   : 0.2,     # InfoNCE temperature for BCL
    "lambda_bcl" : 0.05,    # fixed λ for RecDCL baseline
    "lambda_fcl" : 0.05,    # fixed λ for RecDCL baseline

    # AdaptRecDCL (our proposed method)
    "adapt_alpha": 1.5,     # GradNorm restoring force
    "adapt_lr"   : 0.01,    # LR for learnable λ parameters

    # Per-dataset training config
    "gowalla": {
        "max_epochs" : 200,
        "patience"   : 40,
        "warmup"     : 10,
        "eval_every" : 5,
        "lr_mf"      : 5e-3,
        "lr_lgcn"    : 1e-3,
        # Higher LR for SSL models so they converge faster than MF
        "lr_recdcl"  : 3e-3,
        "lr_adapt"   : 3e-3,
        "n_users_sub": 3000,
    },
    "beauty": {
        "max_epochs" : 300,
        "patience"   : 40,
        "warmup"     : 10,
        "eval_every" : 5,
        "lr_mf"      : 1e-3,
        "lr_lgcn"    : 1e-3,
        "lr_recdcl"  : 1e-3,
        "lr_adapt"   : 1e-3,
        "n_users_sub": None,  # full dataset
    },
}

print("Configuration loaded:")
print(f"  Embedding size : {CFG['emb_size']}")
print(f"  GNN layers     : {CFG['gnn_layers']}")
print(f"  Batch size     : {CFG['batch_size']}")
print(f"  Fixed λ_BCL    : {CFG['lambda_bcl']}  (RecDCL baseline)")
print(f"  Fixed λ_FCL    : {CFG['lambda_fcl']}  (RecDCL baseline)")
print(f"  Adapt α        : {CFG['adapt_alpha']} (AdaptRecDCL)")


Configuration loaded:
  Embedding size : 64
  GNN layers     : 3
  Batch size     : 2048
  Fixed λ_BCL    : 0.05  (RecDCL baseline)
  Fixed λ_FCL    : 0.05  (RecDCL baseline)
  Adapt α        : 1.5 (AdaptRecDCL)


---
## Step 4: Data Loading and Preprocessing

Both datasets use the same preprocessing pipeline:
1. Parse raw interaction data
2. Apply **10-core filtering** (keep users and items with ≥10 interactions) — matches RecDCL paper
3. Split per-user: 80% train / 20% test
4. Build bipartite graph adjacency matrix for GNN models

**Gowalla** is subsampled to 3,000 users for speed (same as Assignment 2).  
**Amazon Beauty** is used in full.


In [4]:
def _download(url, path):
    r = requests.get(url, timeout=300, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    done  = 0
    with open(path, "wb") as f:
        for chunk in r.iter_content(65536):
            f.write(chunk); done += len(chunk)
            if total:
                print(f"\r  {done/1e6:.1f}/{total/1e6:.1f} MB", end="", flush=True)
    print()


def _process(raw, n_users_sub, name):
    """10-core filter, optional subsetting, train/test split."""
    print(f"[DATA] 10-core filtering {name}...")
    prev = -1
    while len(raw) != prev:
        prev = len(raw)
        ic = defaultdict(int)
        for items in raw.values():
            for it in items: ic[it] += 1
        vi  = {it for it, c in ic.items() if c >= 10}
        raw = {u: items & vi for u, items in raw.items()}
        raw = {u: items for u, items in raw.items() if len(items) >= 10}

    if len(raw) == 0:
        raise RuntimeError(f"{name}: dataset empty after 10-core filtering!")

    if n_users_sub:
        users_all = sorted(raw.keys()); random.shuffle(users_all)
        raw = {u: raw[u] for u in users_all[:n_users_sub]}

    users     = sorted(raw.keys())
    all_items = sorted({i for items in raw.values() for i in items})
    u2i  = {u: i for i, u in enumerate(users)}
    it2i = {it: i for i, it in enumerate(all_items)}
    n_u  = len(users); n_i = len(all_items)

    train, test = {}, {}
    for u, items in raw.items():
        items = list(items); random.shuffle(items)
        split = max(1, int(len(items) * 0.8))
        uid   = u2i[u]
        train[uid] = [it2i[i] for i in items[:split]]
        test[uid]  = [it2i[i] for i in items[split:]]

    test  = {u: v for u, v in test.items()  if v}
    train = {u: v for u, v in train.items() if u in test}
    n_train = sum(len(v) for v in train.values())
    # Guard against empty results before computing density
    if n_u == 0 or n_i == 0 or n_train == 0:
        raise RuntimeError(f"{name}: no valid train/test split produced!")
    density = n_train / (n_u * n_i) * 100

    print(f"[DATA] {name}: users={n_u}  items={n_i}  "
          f"train={n_train}  density={density:.4f}%")
    return train, test, n_u, n_i


def load_gowalla(n_users_sub=3000):
    url   = "https://snap.stanford.edu/data/loc-gowalla_totalCheckins.txt.gz"
    cache = "gowalla_full.txt.gz"
    if not os.path.exists(cache):
        print("[DATA] Downloading Gowalla (~105MB)...")
        _download(url, cache)
    print("[DATA] Parsing Gowalla...")
    raw = defaultdict(set)
    with gzip.open(cache, "rt") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 5:
                raw[int(parts[0])].add(parts[4])
    return _process(raw, n_users_sub, "Gowalla")


def load_beauty():
    """
    Amazon Beauty — 5-core dataset (McAuley et al.)
    This is a standard benchmark used in the RecDCL paper.
    Format: user_id,item_id,rating,timestamp  (no header)
    ~52K users, ~57K items after 5-core. Well-behaved with 10-core filter
    since the 5-core pre-filtering ensures sufficient density.
    """
    cache = "beauty_ratings.csv"
    if not os.path.exists(cache):
        print("[DATA] Downloading Amazon Beauty (5-core)...")
        # Primary: McAuley UCSD direct link to 5-core
        urls = [
            "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Beauty.csv",
            "https://jmcauley.ucsd.edu/data/amazon/categoryFiles/ratings_Beauty.csv",
        ]
        downloaded = False
        for url in urls:
            try:
                _download(url, cache)
                downloaded = True
                break
            except Exception as e:
                print(f"[DATA] Failed ({e}), trying next URL...")
        if not downloaded:
            raise RuntimeError("Could not download Amazon Beauty dataset.")

    print("[DATA] Parsing Amazon Beauty...")
    raw = defaultdict(set)
    n_lines = 0
    with open(cache, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 2:
                uid, iid = parts[0].strip(), parts[1].strip()
                # Skip header if present
                if uid.lower() in ("userid", "user_id", "reviewerid"): continue
                if uid and iid:
                    raw[uid].add(iid)
                    n_lines += 1
    print(f"[DATA] Parsed {n_lines:,} raw interactions, {len(raw):,} users")
    if len(raw) == 0:
        raise RuntimeError("Beauty dataset empty after parsing!")
    return _process(raw, None, "Amazon Beauty")


def build_adj(train, n_users, n_items):
    rows, cols = [], []
    for u, items in train.items():
        for i in items:
            rows.append(u);           cols.append(n_users + i)
            rows.append(n_users + i); cols.append(u)
    N   = n_users + n_items
    deg = torch.zeros(N)
    for r in rows: deg[r] += 1
    d   = deg.pow(-0.5); d[d == float("inf")] = 0
    adj = torch.sparse_coo_tensor(
        torch.tensor([rows, cols], dtype=torch.long),
        d[rows] * d[cols], (N, N)
    ).coalesce().to(DEVICE)
    return adj


def get_batch(train, n_items, batch_size):
    users = list(train.keys()); random.shuffle(users)
    for i in range(0, len(users), batch_size):
        bu  = users[i: i + batch_size]
        pos = [random.choice(train[u]) for u in bu]
        neg = []
        for u in bu:
            n = random.randint(0, n_items - 1)
            while n in train[u]: n = random.randint(0, n_items - 1)
            neg.append(n)
        yield (torch.tensor(bu,  device=DEVICE),
               torch.tensor(pos, device=DEVICE),
               torch.tensor(neg, device=DEVICE))

print("✓ Data utilities defined")
print("  Datasets available: Gowalla (subset), Amazon Beauty")


✓ Data utilities defined
  Datasets available: Gowalla (subset), Amazon Beauty


---
## Step 5: Model Definitions

### MF — Matrix Factorization
Standard collaborative filtering with BPR (Bayesian Personalised Ranking) loss.  
No graph structure. Serves as the weakest baseline.

### LightGCN
Propagates embeddings over the user-item bipartite graph across L layers.  
Mean-pools layer outputs. Strong graph-based baseline.

### RecDCL (Assignment 2 paper)
Builds on LightGCN encoder with two auxiliary contrastive objectives:
- **BCL**: Batch-wise InfoNCE — (user, pos_item) pairs as positives
- **FCL**: Feature-wise — Gram matrix orthogonality across embedding dimensions  
Fixed λ_BCL = λ_FCL = 0.05.

### AdaptRecDCL (Our Proposed Method)
Identical architecture to RecDCL but λ_BCL and λ_FCL are **learnable parameters** updated via **GradNorm**:

> *GradNorm (Chen et al., ICML 2018) monitors the gradient magnitude each task places on shared parameters. It then adjusts task weights so that no single loss dominates gradient flow — keeping all tasks balanced throughout training.*

**Why this matters:** In A2, fixed λ caused RecDCL to collapse on the full Gowalla dataset because the InfoNCE loss produced gradients ≈7× larger than BPR. AdaptRecDCL detects and corrects this imbalance automatically.


In [5]:
# ─────────────────────────────────────────────────────────────
# MF
# ─────────────────────────────────────────────────────────────
class MF(nn.Module):
    name = "MF"
    def __init__(self, n_u, n_i, d):
        super().__init__()
        self.U = nn.Embedding(n_u, d); self.I = nn.Embedding(n_i, d)
        nn.init.xavier_uniform_(self.U.weight)
        nn.init.xavier_uniform_(self.I.weight)

    def forward(self, u, p, n):
        eu = self.U(u); ep = self.I(p); en = self.I(n)
        bpr = -F.logsigmoid((eu*ep).sum(-1) - (eu*en).sum(-1)).mean()
        l2  = (eu.norm(2).pow(2)+ep.norm(2).pow(2)+en.norm(2).pow(2)) / len(u)
        return bpr + CFG["l2_reg"]*l2, bpr.item()

    @torch.no_grad()
    def get_all_embeddings(self): return self.U.weight, self.I.weight


# ─────────────────────────────────────────────────────────────
# LightGCN
# ─────────────────────────────────────────────────────────────
class LightGCN(nn.Module):
    name = "LightGCN"
    def __init__(self, n_u, n_i, d, adj, L):
        super().__init__()
        self.n_u = n_u; self.adj = adj; self.L = L
        self.U = nn.Embedding(n_u, d); self.I = nn.Embedding(n_i, d)
        nn.init.xavier_uniform_(self.U.weight)
        nn.init.xavier_uniform_(self.I.weight)

    def propagate(self):
        e = torch.cat([self.U.weight, self.I.weight]); out = [e]
        for _ in range(self.L):
            e = torch.sparse.mm(self.adj, e); out.append(e)
        e = torch.stack(out).mean(0)
        return e[:self.n_u], e[self.n_u:]

    def forward(self, u, p, n):
        ua, ia = self.propagate()
        eu=ua[u]; ep=ia[p]; en=ia[n]
        bpr = -F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        u0=self.U(u); p0=self.I(p); n0=self.I(n)
        l2 = (u0.norm(2).pow(2)+p0.norm(2).pow(2)+n0.norm(2).pow(2))/len(u)
        return bpr+CFG["l2_reg"]*l2, bpr.item()

    @torch.no_grad()
    def get_all_embeddings(self): return self.propagate()


# ─────────────────────────────────────────────────────────────
# RecDCL — Fixed λ (Assignment 2 paper baseline)
# ─────────────────────────────────────────────────────────────
class RecDCL(nn.Module):
    name = "RecDCL"
    def __init__(self, n_u, n_i, d, adj, L):
        super().__init__()
        self.n_u=n_u; self.adj=adj; self.L=L
        self.U=nn.Embedding(n_u,d); self.I=nn.Embedding(n_i,d)
        self.proj=nn.Linear(d,d,bias=False)
        nn.init.xavier_uniform_(self.U.weight)
        nn.init.xavier_uniform_(self.I.weight)
        nn.init.xavier_uniform_(self.proj.weight)

    def propagate(self):
        e=torch.cat([self.U.weight,self.I.weight]); out=[e]
        for _ in range(self.L):
            e=torch.sparse.mm(self.adj,e); out.append(e)
        e=torch.stack(out).mean(0)
        return e[:self.n_u], e[self.n_u:]

    def bcl(self, u, i):
        zu=F.normalize(self.proj(u),dim=-1)
        zi=F.normalize(self.proj(i),dim=-1)
        logits=zu@zi.T/CFG["bcl_temp"]
        labels=torch.arange(len(zu),device=DEVICE)
        return (F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))/2

    def fcl(self, u, i):
        align=(F.normalize(u,dim=-1)*F.normalize(i,dim=-1)).sum(-1).mean()
        nu=(u-u.mean(0))/(u.std(0)+1e-8)
        ni=(i-i.mean(0))/(i.std(0)+1e-8)
        c=nu.T@ni/len(u)
        eye=torch.eye(c.size(0),device=DEVICE)
        return -align+0.5*((c-eye).pow(2)).sum()

    def forward(self, u, p, n):
        ua,ia=self.propagate()
        eu=ua[u]; ep=ia[p]; en=ia[n]
        bpr=-F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        bcl=self.bcl(eu,ep); fcl=self.fcl(eu,ep)
        u0=self.U(u); p0=self.I(p); n0=self.I(n)
        l2=(u0.norm(2).pow(2)+p0.norm(2).pow(2)+n0.norm(2).pow(2))/len(u)
        return bpr+CFG["lambda_bcl"]*bcl+CFG["lambda_fcl"]*fcl+CFG["l2_reg"]*l2, bpr.item()

    @torch.no_grad()
    def get_all_embeddings(self): return self.propagate()


# ─────────────────────────────────────────────────────────────
# AdaptRecDCL — Our Proposed Method
# Reference: Chen et al., "GradNorm: Gradient Normalization for
# Adaptive Loss Balancing in Deep Multitask Networks", ICML 2018
# ─────────────────────────────────────────────────────────────
class AdaptRecDCL(nn.Module):
    """
    RecDCL with GradNorm-based adaptive loss weighting.

    Key idea: λ_BCL and λ_FCL are learnable log-scale parameters.
    A GradNorm auxiliary loss penalises any task whose gradient norm
    deviates from the balanced target, and is used to update λ via
    a separate small-LR optimiser every 5 training steps.

    This eliminates manual per-dataset λ tuning and prevents
    the contrastive loss collapse observed in A2 full-scale runs.
    """
    name = "AdaptRecDCL"

    def __init__(self, n_u, n_i, d, adj, L):
        super().__init__()
        self.n_u=n_u; self.adj=adj; self.L=L
        self.U=nn.Embedding(n_u,d); self.I=nn.Embedding(n_i,d)
        self.proj=nn.Linear(d,d,bias=False)
        nn.init.xavier_uniform_(self.U.weight)
        nn.init.xavier_uniform_(self.I.weight)
        nn.init.xavier_uniform_(self.proj.weight)

        # Learnable log-weights — log space ensures λ > 0 always
        # Initialised to log(0.05) ≈ -3.0 (same starting point as RecDCL)
        self.log_lam_bcl = nn.Parameter(torch.tensor(-3.0))
        self.log_lam_fcl = nn.Parameter(torch.tensor(-3.0))

        # Reference losses for GradNorm (set on first forward pass)
        self._l0_bpr = None; self._l0_bcl = None; self._l0_fcl = None

    @property
    def lambda_bcl(self): return self.log_lam_bcl.exp()

    @property
    def lambda_fcl(self): return self.log_lam_fcl.exp()

    def propagate(self):
        e=torch.cat([self.U.weight,self.I.weight]); out=[e]
        for _ in range(self.L):
            e=torch.sparse.mm(self.adj,e); out.append(e)
        e=torch.stack(out).mean(0)
        return e[:self.n_u], e[self.n_u:]

    def bcl(self, u, i):
        zu=F.normalize(self.proj(u),dim=-1)
        zi=F.normalize(self.proj(i),dim=-1)
        logits=zu@zi.T/CFG["bcl_temp"]
        labels=torch.arange(len(zu),device=DEVICE)
        return (F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))/2

    def fcl(self, u, i):
        align=(F.normalize(u,dim=-1)*F.normalize(i,dim=-1)).sum(-1).mean()
        nu=(u-u.mean(0))/(u.std(0)+1e-8)
        ni=(i-i.mean(0))/(i.std(0)+1e-8)
        c=nu.T@ni/len(u)
        eye=torch.eye(c.size(0),device=DEVICE)
        return -align+0.5*((c-eye).pow(2)).sum()

    def forward(self, u, p, n):
        ua,ia=self.propagate()
        eu=ua[u]; ep=ia[p]; en=ia[n]
        bpr_l=-F.logsigmoid((eu*ep).sum(-1)-(eu*en).sum(-1)).mean()
        bcl_l=self.bcl(eu,ep)
        fcl_l=self.fcl(eu,ep)
        u0=self.U(u); p0=self.I(p); n0=self.I(n)
        l2=(u0.norm(2).pow(2)+p0.norm(2).pow(2)+n0.norm(2).pow(2))/len(u)
        total=bpr_l+self.lambda_bcl*bcl_l+self.lambda_fcl*fcl_l+CFG["l2_reg"]*l2
        return total, bpr_l.item(), bcl_l.detach(), fcl_l.detach()

    def gradnorm_loss(self, bpr_l, bcl_l, fcl_l):
        """
        Compute GradNorm auxiliary loss for λ update.
        Projects each task's loss gradient onto the shared projection
        layer, then penalises deviation from balanced gradient norms.
        """
        shared = list(self.proj.parameters())[0]

        def gnorm(loss):
            g = torch.autograd.grad(loss, shared,
                                    retain_graph=True, create_graph=True,
                                    allow_unused=True)[0]
            # If loss has no path through shared param, return small constant
            if g is None:
                return torch.tensor(1e-8, device=DEVICE, requires_grad=True)
            return g.norm(2) + 1e-8  # eps avoids zero norm instability

        G_bpr = gnorm(bpr_l)
        G_bcl = gnorm(bcl_l * self.lambda_bcl)
        G_fcl = gnorm(fcl_l * self.lambda_fcl)
        G_avg = (G_bpr + G_bcl + G_fcl).detach() / 3

        # Initialise reference losses
        if self._l0_bpr is None:
            self._l0_bpr=bpr_l.detach()
            self._l0_bcl=bcl_l.detach()
            self._l0_fcl=fcl_l.detach()

        alpha = CFG["adapt_alpha"]
        eps   = 1e-8
        r_bpr = bpr_l.detach()/(self._l0_bpr+eps)
        r_bcl = bcl_l.detach()/(self._l0_bcl+eps)
        r_fcl = fcl_l.detach()/(self._l0_fcl+eps)
        r_avg = (r_bpr+r_bcl+r_fcl)/3

        T_bpr = (G_avg*(r_bpr/r_avg)**alpha).detach()
        T_bcl = (G_avg*(r_bcl/r_avg)**alpha).detach()
        T_fcl = (G_avg*(r_fcl/r_avg)**alpha).detach()

        return (torch.abs(G_bpr-T_bpr)+
                torch.abs(G_bcl-T_bcl)+
                torch.abs(G_fcl-T_fcl))

    @torch.no_grad()
    def get_all_embeddings(self): return self.propagate()


print("✓ All model classes defined: MF, LightGCN, RecDCL, AdaptRecDCL")


✓ All model classes defined: MF, LightGCN, RecDCL, AdaptRecDCL


---
## Step 6: Evaluation Function

Full ranking evaluation — for each test user, score all items not seen during training,
take top-20, and compute **Recall@20** and **NDCG@20**.

This matches the evaluation protocol used in the RecDCL paper.


In [6]:
@torch.no_grad()
def evaluate(model, train, test, topk=20):
    model.eval()
    u_emb, i_emb = model.get_all_embeddings()
    recalls, ndcgs = [], []
    test_users = list(test.keys())
    for start in range(0, len(test_users), 512):
        bu     = test_users[start: start+512]
        scores = u_emb[bu] @ i_emb.T
        for j, u in enumerate(bu):
            if train.get(u): scores[j, train[u]] = -1e9
        _, top_k = scores.topk(topk, dim=1)
        top_k = top_k.cpu().numpy()
        for j, u in enumerate(bu):
            gt   = set(test[u]); pred = top_k[j].tolist()
            hits = [1 if p in gt else 0 for p in pred]
            recalls.append(sum(hits)/max(len(gt),1))
            dcg  = sum(h/math.log2(r+2) for r,h in enumerate(hits))
            idcg = sum(1/math.log2(r+2) for r in range(min(len(gt),topk)))
            ndcgs.append(dcg/idcg if idcg>0 else 0)
    model.train()
    return float(np.mean(recalls)), float(np.mean(ndcgs))

print("✓ Evaluation function defined")


✓ Evaluation function defined


---
## Step 7: Training Loops

Two training loops:

1. **`train_standard`** — for MF, LightGCN, RecDCL. Single Adam optimiser, early stopping.

2. **`train_adapt`** — for AdaptRecDCL. Two optimisers:
   - `main_opt`: updates embeddings and projection head (standard)
   - `adapt_opt`: updates only `log_λ_BCL` and `log_λ_FCL` using GradNorm loss every 5 steps

The GradNorm update frequency of 5 steps is a balance between responsiveness and stability.


In [7]:
def train_standard(model, lr, train, test, n_items, dcfg, dataset_name):
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"epoch":[],"loss":[],"recall":[],"ndcg":[],
               "dataset":dataset_name,"model":model.name}
    best_recall=best_ndcg=best_ep=no_imp=0

    print(f"\n{'─'*60}")
    print(f"  [{dataset_name}] {model.name}  lr={lr}  max_ep={dcfg['max_epochs']}")
    print(f"{'─'*60}")
    t0=time.time()

    for epoch in range(1, dcfg["max_epochs"]+1):
        model.train(); tl=nb=0
        for u,p,n in get_batch(train, n_items, CFG["batch_size"]):
            opt.zero_grad()
            loss,_ = model(u,p,n)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); tl+=loss.item(); nb+=1

        if epoch%dcfg["eval_every"]==0 or epoch==1:
            recall,ndcg = evaluate(model,train,test)
            imp = recall>best_recall
            if imp:
                best_recall,best_ndcg,best_ep,no_imp=recall,ndcg,epoch,0
                torch.save(model.state_dict(),
                           f"results_a3/{dataset_name}_{model.name}_best.pt")
            else: no_imp+=1
            history["epoch"].append(epoch)
            history["loss"].append(round(tl/max(nb,1),5))
            history["recall"].append(round(recall,5))
            history["ndcg"].append(round(ndcg,5))
            print(f"  Ep{epoch:>4}  Loss:{tl/nb:.4f}  R@20:{recall:.4f}  "
                  f"N@20:{ndcg:.4f}  [{time.time()-t0:.0f}s]{'  ◀' if imp else ''}")
            if epoch>dcfg.get("warmup",20) and no_imp>=dcfg["patience"]:
                print(f"  Early stop @ ep{epoch}"); break

    print(f"\n  ✓ {model.name}  Best R@20={best_recall:.4f}  N@20={best_ndcg:.4f}  @ep{best_ep}")
    history.update({"best_recall":best_recall,"best_ndcg":best_ndcg,"best_epoch":best_ep})
    return history


def train_adapt(model, lr, adapt_lr, train, test, n_items, dcfg, dataset_name):
    model = model.to(DEVICE)
    lambda_params = [model.log_lam_bcl, model.log_lam_fcl]
    model_params  = [p for p in model.parameters()
                     if p is not model.log_lam_bcl and p is not model.log_lam_fcl]
    main_opt  = torch.optim.Adam(model_params,  lr=lr)
    adapt_opt = torch.optim.Adam(lambda_params, lr=adapt_lr)

    history = {"epoch":[],"loss":[],"recall":[],"ndcg":[],
               "lambda_bcl":[],"lambda_fcl":[],
               "dataset":dataset_name,"model":model.name}
    best_recall=best_ndcg=best_ep=no_imp=0; step=0

    print(f"\n{'─'*60}")
    print(f"  [{dataset_name}] {model.name}  lr={lr}  adapt_lr={adapt_lr}")
    print(f"{'─'*60}")
    t0=time.time()

    for epoch in range(1, dcfg["max_epochs"]+1):
        model.train(); tl=nb=0
        ep_lbcl=[]; ep_lfcl=[]

        for u,p,n in get_batch(train, n_items, CFG["batch_size"]):
            step+=1
            # Main update
            main_opt.zero_grad(); adapt_opt.zero_grad()
            total,_,bcl_d,fcl_d = model(u,p,n)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model_params,1.0)
            main_opt.step()

            # GradNorm update every 5 steps
            if step%5==0:
                main_opt.zero_grad(); adapt_opt.zero_grad()
                ua,ia=model.propagate()
                eu=ua[u]; ep_e=ia[p]; en=ia[n]
                bpr_l=-F.logsigmoid((eu*ep_e).sum(-1)-(eu*en).sum(-1)).mean()
                bcl_l=model.bcl(eu,ep_e); fcl_l=model.fcl(eu,ep_e)
                try:
                    gn=model.gradnorm_loss(bpr_l,bcl_l,fcl_l)
                    gn.backward(); adapt_opt.step()
                except Exception:
                    pass  # skip gradnorm step if graph unavailable
                with torch.no_grad():
                    model.log_lam_bcl.clamp_(-9.2,0.0)
                    model.log_lam_fcl.clamp_(-9.2,0.0)

            tl+=total.item(); nb+=1
            ep_lbcl.append(model.lambda_bcl.item())
            ep_lfcl.append(model.lambda_fcl.item())

        if epoch%dcfg["eval_every"]==0 or epoch==1:
            recall,ndcg=evaluate(model,train,test)
            imp=recall>best_recall
            if imp:
                best_recall,best_ndcg,best_ep,no_imp=recall,ndcg,epoch,0
                torch.save(model.state_dict(),
                           f"results_a3/{dataset_name}_{model.name}_best.pt")
            else: no_imp+=1
            lb=float(np.mean(ep_lbcl)); lf=float(np.mean(ep_lfcl))
            history["epoch"].append(epoch)
            history["loss"].append(round(tl/max(nb,1),5))
            history["recall"].append(round(recall,5))
            history["ndcg"].append(round(ndcg,5))
            history["lambda_bcl"].append(round(lb,5))
            history["lambda_fcl"].append(round(lf,5))
            print(f"  Ep{epoch:>4}  Loss:{tl/nb:.4f}  R@20:{recall:.4f}  "
                  f"N@20:{ndcg:.4f}  λ_B:{lb:.4f}  λ_F:{lf:.4f}"
                  f"  [{time.time()-t0:.0f}s]{'  ◀' if imp else ''}")
            if epoch>dcfg.get("warmup",20) and no_imp>=dcfg["patience"]:
                print(f"  Early stop @ ep{epoch}"); break

    print(f"\n  ✓ {model.name}  Best R@20={best_recall:.4f}  N@20={best_ndcg:.4f}  @ep{best_ep}")
    print(f"  Final λ_BCL={model.lambda_bcl.item():.4f}  λ_FCL={model.lambda_fcl.item():.4f}")
    history.update({"best_recall":best_recall,"best_ndcg":best_ndcg,"best_epoch":best_ep,
                    "final_lam_bcl":model.lambda_bcl.item(),
                    "final_lam_fcl":model.lambda_fcl.item()})
    return history

print("✓ Training loops defined")


✓ Training loops defined


---
## Step 8: Plotting Utilities

Three plot types:
1. **Training curves** — loss, Recall@20, NDCG@20 per epoch for all 4 models
2. **Lambda evolution** — how λ_BCL and λ_FCL adapt over time in AdaptRecDCL
3. **Cross-dataset bar chart** — final performance comparison across both datasets


In [8]:
COLORS = {"MF":"#e05c3a","LightGCN":"#2e6bbf",
          "RecDCL":"#1a9e6e","AdaptRecDCL":"#9b30d9"}

def plot_training_curves(histories, dataset_name):
    fig, axes = plt.subplots(1, 3, figsize=(16,5))
    fig.suptitle(f"Training Curves — {dataset_name}\n"
                 "MF vs LightGCN vs RecDCL vs AdaptRecDCL (Proposed)",
                 fontsize=12, fontweight="bold")
    for ax,(key,title) in zip(axes,[("loss","Training Loss"),
                                    ("recall","Recall@20"),("ndcg","NDCG@20")]):
        for h in histories:
            m=h["model"]
            ax.plot(h["epoch"],h[key],label=m,color=COLORS.get(m,"gray"),
                    linewidth=2.5 if m=="AdaptRecDCL" else 2.0,
                    linestyle="--" if m=="AdaptRecDCL" else "-",
                    marker="o",markersize=3)
        ax.set_title(title,fontweight="bold"); ax.set_xlabel("Epoch")
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    path=f"results_a3/curves_{dataset_name.replace(' ','_')}.png"
    plt.savefig(path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"[PLOT] {path}")


def plot_lambda_evolution(h, dataset_name):
    fig,ax=plt.subplots(figsize=(9,4))
    ax.plot(h["epoch"],h["lambda_bcl"],label="λ_BCL (adaptive)",
            color="#e05c3a",linewidth=2)
    ax.plot(h["epoch"],h["lambda_fcl"],label="λ_FCL (adaptive)",
            color="#2e6bbf",linewidth=2)
    ax.axhline(y=0.05,color="gray",linestyle="--",linewidth=1.2,
               label="RecDCL fixed λ = 0.05")
    ax.set_xlabel("Epoch"); ax.set_ylabel("λ value")
    ax.set_title(f"AdaptRecDCL — Adaptive Lambda Evolution ({dataset_name})",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    path=f"results_a3/lambda_evolution_{dataset_name.replace(' ','_')}.png"
    plt.savefig(path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"[PLOT] {path}")


def plot_cross_dataset(all_results):
    models  =["MF","LightGCN","RecDCL","AdaptRecDCL"]
    datasets=list(all_results.keys())
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    fig.suptitle("Cross-Dataset Performance — Recall@20 and NDCG@20",
                 fontsize=13,fontweight="bold")
    x=np.arange(len(models)); w=0.35
    offs=[-w/2,w/2]
    HATCH=["","///"]
    for ax,metric in zip(axes,["recall","ndcg"]):
        for di,(dname,off,hat) in enumerate(zip(datasets,offs,HATCH)):
            vals=[all_results[dname].get(m,{}).get(f"best_{metric}",0) for m in models]
            bars=ax.bar(x+off,vals,w,label=dname,alpha=0.85,hatch=hat,
                        color=[COLORS[m] for m in models],
                        edgecolor="white",linewidth=0.5)
            for bar,v in zip(bars,vals):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.001,
                        f"{v:.3f}",ha="center",va="bottom",fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(models,fontsize=10)
        ax.set_ylabel(f"{'Recall' if metric=='recall' else 'NDCG'}@20")
        ax.set_title(f"{'Recall' if metric=='recall' else 'NDCG'}@20")
        ax.legend(); ax.grid(axis="y",alpha=0.3)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    path="results_a3/cross_dataset_comparison.png"
    plt.savefig(path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"[PLOT] {path}")

print("✓ Plotting utilities defined")


✓ Plotting utilities defined


---
## Step 9: Experiment 1 — Gowalla Subset

Replicates the Assignment 2 baseline on the same 3,000-user Gowalla subset,
now including **AdaptRecDCL** as a fourth model.

This serves as the within-dataset comparison and validates that AdaptRecDCL
improves on RecDCL even where RecDCL already worked.


In [9]:
print("Loading Gowalla subset (3,000 users)...")
g_train, g_test, g_nu, g_ni = load_gowalla(n_users_sub=3000)
g_adj = build_adj(g_train, g_nu, g_ni)
d=CFG["emb_size"]; L=CFG["gnn_layers"]; dcfg=CFG["gowalla"]

g_histories = []
g_results   = {}

for name, model, lr, adapt_lr in [
    ("MF",          MF(g_nu,g_ni,d),                       dcfg["lr_mf"],     None),
    ("LightGCN",    LightGCN(g_nu,g_ni,d,g_adj,L),         dcfg["lr_lgcn"],   None),
    ("RecDCL",      RecDCL(g_nu,g_ni,d,g_adj,L),           dcfg["lr_recdcl"], None),
    ("AdaptRecDCL", AdaptRecDCL(g_nu,g_ni,d,g_adj,L),      dcfg["lr_adapt"],  dcfg["lr_adapt"]),
]:
    if adapt_lr:
        h = train_adapt(model,lr,adapt_lr,g_train,g_test,g_ni,dcfg,"Gowalla")
        plot_lambda_evolution(h,"Gowalla")
    else:
        h = train_standard(model,lr,g_train,g_test,g_ni,dcfg,"Gowalla")
    g_histories.append(h)
    g_results[name]={"best_recall":h["best_recall"],
                     "best_ndcg":h["best_ndcg"],
                     "best_epoch":h["best_epoch"]}

plot_training_curves(g_histories,"Gowalla")
print("\n✓ Gowalla experiments complete")


Loading Gowalla subset (3,000 users)...
[DATA] Downloading Gowalla (~105MB)...
  105.5/105.5 MB
[DATA] Parsing Gowalla...
[DATA] 10-core filtering Gowalla...
[DATA] Gowalla: users=3000  items=33413  train=83575  density=0.0834%

────────────────────────────────────────────────────────────
  [Gowalla] MF  lr=0.005  max_ep=200
────────────────────────────────────────────────────────────
  Ep   1  Loss:0.6931  R@20:0.0012  N@20:0.0007  [2s]  ◀
  Ep   5  Loss:0.6910  R@20:0.0012  N@20:0.0007  [3s]
  Ep  10  Loss:0.6857  R@20:0.0011  N@20:0.0007  [3s]
  Ep  15  Loss:0.6770  R@20:0.0022  N@20:0.0014  [4s]  ◀
  Ep  20  Loss:0.6641  R@20:0.0030  N@20:0.0019  [4s]  ◀
  Ep  25  Loss:0.6439  R@20:0.0046  N@20:0.0029  [5s]  ◀
  Ep  30  Loss:0.6193  R@20:0.0065  N@20:0.0049  [6s]  ◀
  Ep  35  Loss:0.5870  R@20:0.0101  N@20:0.0076  [8s]  ◀
  Ep  40  Loss:0.5534  R@20:0.0132  N@20:0.0105  [8s]  ◀
  Ep  45  Loss:0.5151  R@20:0.0167  N@20:0.0134  [9s]  ◀
  Ep  50  Loss:0.4776  R@20:0.0197  N@20:0.0164 

### Gowalla Results Summary

In [10]:
print("\n" + "="*62)
print("  GOWALLA RESULTS")
print("="*62)
print(f"  {'Model':<16} {'Recall@20':>10} {'NDCG@20':>10} {'vs MF':>10} {'Best Ep':>8}")
print(f"  {'─'*56}")
mf_r = g_results["MF"]["best_recall"]
for m,v in g_results.items():
    imp = f"+{(v['best_recall']-mf_r)/mf_r*100:.1f}%" if m!="MF" else "baseline"
    print(f"  {m:<16} {v['best_recall']:>10.4f} {v['best_ndcg']:>10.4f} "
          f"{imp:>10} {v['best_epoch']:>8}")

# Show adaptive lambda final values
for h in g_histories:
    if h["model"]=="AdaptRecDCL":
        print(f"\n  AdaptRecDCL final λ_BCL = {h.get('final_lam_bcl',0):.4f}")
        print(f"  AdaptRecDCL final λ_FCL = {h.get('final_lam_fcl',0):.4f}")
        print(f"  (RecDCL fixed λ = 0.0500 for comparison)")



  GOWALLA RESULTS
  Model             Recall@20    NDCG@20      vs MF  Best Ep
  ────────────────────────────────────────────────────────
  MF                   0.0662     0.0579   baseline      200
  LightGCN             0.0962     0.0779     +45.3%       35
  RecDCL               0.0537     0.0355    +-18.8%      165
  AdaptRecDCL          0.0525     0.0351    +-20.6%      200

  AdaptRecDCL final λ_BCL = 0.0405
  AdaptRecDCL final λ_FCL = 0.0498
  (RecDCL fixed λ = 0.0500 for comparison)


---
## Step 10: Experiment 2 — Amazon Beauty (Mandatory New Dataset)

Amazon Beauty is a product review dataset from McAuley et al., widely used as a benchmark
in self-supervised recommendation papers including RecDCL. It is a **new dataset** not used
in Assignment 2, satisfying the mandatory cross-dataset requirement.

**Why Beauty instead of Sports?**
Amazon Sports has ~1.9M users in its raw form. After 10-core filtering (which iteratively
removes users and items with fewer than 10 interactions until stable), the dataset converges
to a very small residual that causes division-by-zero errors in density computation.
Amazon Beauty (~52K users, ~57K items, well-balanced) is a standard RecDCL benchmark
that handles 10-core filtering cleanly.

All 4 models are trained on the full dataset with early stopping (patience=40).


In [11]:
print("Loading Amazon Beauty...")
s_train, s_test, s_nu, s_ni = load_beauty()

# Verify dataset loaded correctly
assert s_nu > 0 and s_ni > 0, "Beauty dataset failed to load!"
n_train_s = sum(len(v) for v in s_train.values())
assert n_train_s > 0, "No training interactions!"
print(f"\n✓ Beauty verified: {s_nu} users, {s_ni} items, "
      f"{n_train_s:,} train interactions")

s_adj  = build_adj(s_train, s_nu, s_ni)
dcfg_s = CFG["beauty"]
d      = CFG["emb_size"]
L      = CFG["gnn_layers"]

s_histories = []
s_results   = {}

for name, model, lr, adapt_lr in [
    ("MF",          MF(s_nu, s_ni, d),                        dcfg_s["lr_mf"],     None),
    ("LightGCN",    LightGCN(s_nu, s_ni, d, s_adj, L),        dcfg_s["lr_lgcn"],   None),
    ("RecDCL",      RecDCL(s_nu, s_ni, d, s_adj, L),          dcfg_s["lr_recdcl"], None),
    ("AdaptRecDCL", AdaptRecDCL(s_nu, s_ni, d, s_adj, L),     dcfg_s["lr_adapt"],  dcfg_s["lr_adapt"]),
]:
    if adapt_lr is not None:
        h = train_adapt(model, lr, adapt_lr, s_train, s_test,
                        s_ni, dcfg_s, "Amazon_Beauty")
        plot_lambda_evolution(h, "Amazon_Beauty")
    else:
        h = train_standard(model, lr, s_train, s_test,
                           s_ni, dcfg_s, "Amazon_Beauty")
    s_histories.append(h)
    s_results[name] = {
        "best_recall": h["best_recall"],
        "best_ndcg":   h["best_ndcg"],
        "best_epoch":  h["best_epoch"],
    }

plot_training_curves(s_histories, "Amazon_Beauty")
print("\n✓ Amazon Beauty experiments complete")


Loading Amazon Beauty...
[DATA] Downloading Amazon Beauty (5-core)...
  82.4/82.4 MB
[DATA] Parsing Amazon Beauty...
[DATA] Parsed 2,023,070 raw interactions, 1,210,271 users
[DATA] 10-core filtering Amazon Beauty...
[DATA] Amazon Beauty: users=1340  items=733  train=22492  density=2.2899%

✓ Beauty verified: 1340 users, 733 items, 22,492 train interactions

────────────────────────────────────────────────────────────
  [Amazon_Beauty] MF  lr=0.001  max_ep=300
────────────────────────────────────────────────────────────
  Ep   1  Loss:0.6931  R@20:0.0266  N@20:0.0153  [0s]  ◀
  Ep   5  Loss:0.6930  R@20:0.0264  N@20:0.0154  [0s]
  Ep  10  Loss:0.6923  R@20:0.0260  N@20:0.0152  [0s]
  Ep  15  Loss:0.6914  R@20:0.0245  N@20:0.0148  [0s]
  Ep  20  Loss:0.6911  R@20:0.0259  N@20:0.0153  [1s]
  Ep  25  Loss:0.6898  R@20:0.0259  N@20:0.0155  [1s]
  Ep  30  Loss:0.6892  R@20:0.0255  N@20:0.0155  [1s]
  Ep  35  Loss:0.6889  R@20:0.0258  N@20:0.0155  [1s]
  Ep  40  Loss:0.6883  R@20:0.0261  N@2

### Amazon Beauty Results Summary

In [12]:
print("\n" + "="*62)
print("  AMAZON BEAUTY RESULTS")
print("="*62)
print(f"  {'Model':<16} {'Recall@20':>10} {'NDCG@20':>10} {'vs MF':>10} {'Best Ep':>8}")
print(f"  {'-'*56}")
mf_r = s_results["MF"]["best_recall"]
for m, v in s_results.items():
    imp = f"+{(v['best_recall']-mf_r)/max(mf_r,1e-9)*100:.1f}%" if m != "MF" else "baseline"
    print(f"  {m:<16} {v['best_recall']:>10.4f} {v['best_ndcg']:>10.4f} "
          f"{imp:>10} {v['best_epoch']:>8}")

for h in s_histories:
    if h["model"] == "AdaptRecDCL":
        print(f"\n  AdaptRecDCL final λ_BCL = {h.get('final_lam_bcl', 0):.4f}")
        print(f"  AdaptRecDCL final λ_FCL = {h.get('final_lam_fcl', 0):.4f}")
        print(f"  (RecDCL fixed λ = 0.0500 for comparison)")



  AMAZON BEAUTY RESULTS
  Model             Recall@20    NDCG@20      vs MF  Best Ep
  --------------------------------------------------------
  MF                   0.2328     0.1637   baseline      300
  LightGCN             0.2334     0.1636      +0.2%       60
  RecDCL               0.1929     0.1417    +-17.2%      300
  AdaptRecDCL          0.1855     0.1334    +-20.4%      285

  AdaptRecDCL final λ_BCL = 0.0471
  AdaptRecDCL final λ_FCL = 0.0498
  (RecDCL fixed λ = 0.0500 for comparison)


---
## Step 11: Cross-Dataset Comparison

Final bar chart and consolidated table comparing all models across both datasets.  
This is the key visualisation for the report — shows whether findings generalise.


In [13]:
all_results = {"Gowalla": g_results, "Amazon_Beauty": s_results}
plot_cross_dataset(all_results)

print("\n" + "="*70)
print("  FINAL RESULTS — ALL DATASETS")
print("="*70)
for dname, dresults in all_results.items():
    print(f"\n  {dname}:")
    print(f"  {'Model':<16} {'Recall@20':>10} {'NDCG@20':>10} {'vs MF':>10} {'Best Ep':>8}")
    print(f"  {'-'*58}")
    mf_r = dresults["MF"]["best_recall"]
    for m, v in dresults.items():
        imp = f"+{(v['best_recall']-mf_r)/max(mf_r,1e-9)*100:.1f}%" if m != "MF" else "baseline"
        print(f"  {m:<16} {v['best_recall']:>10.4f} {v['best_ndcg']:>10.4f} "
              f"{imp:>10} {v['best_epoch']:>8}")


[PLOT] results_a3/cross_dataset_comparison.png

  FINAL RESULTS — ALL DATASETS

  Gowalla:
  Model             Recall@20    NDCG@20      vs MF  Best Ep
  ----------------------------------------------------------
  MF                   0.0662     0.0579   baseline      200
  LightGCN             0.0962     0.0779     +45.3%       35
  RecDCL               0.0537     0.0355    +-18.8%      165
  AdaptRecDCL          0.0525     0.0351    +-20.6%      200

  Amazon_Beauty:
  Model             Recall@20    NDCG@20      vs MF  Best Ep
  ----------------------------------------------------------
  MF                   0.2328     0.1637   baseline      300
  LightGCN             0.2334     0.1636      +0.2%       60
  RecDCL               0.1929     0.1417    +-17.2%      300
  AdaptRecDCL          0.1855     0.1334    +-20.4%      285


---
## Step 12: Save All Results

Saves a JSON file with all numbers for the report, and lists all generated plots.


In [14]:
# Save JSON
out = {
    "config":             CFG,
    "results":            all_results,
    "gowalla_histories":  [{k: v for k, v in h.items()
                            if k not in ("dataset", "model")}
                           for h in g_histories],
    "beauty_histories":   [{k: v for k, v in h.items()
                            if k not in ("dataset", "model")}
                           for h in s_histories],
}
with open("results_a3/all_results.json", "w") as f:
    json.dump(out, f, indent=2)
print("Saved: results_a3/all_results.json")

print("\nGenerated files:")
for fname in sorted(os.listdir("results_a3")):
    size = os.path.getsize(f"results_a3/{fname}")
    print(f"  {fname:<52} {size/1024:.1f} KB")


Saved: results_a3/all_results.json

Generated files:
  Amazon_Beauty_AdaptRecDCL_best.pt                    537.3 KB
  Amazon_Beauty_LightGCN_best.pt                       520.4 KB
  Amazon_Beauty_MF_best.pt                             520.3 KB
  Amazon_Beauty_RecDCL_best.pt                         536.7 KB
  Gowalla_AdaptRecDCL_best.pt                          9122.2 KB
  Gowalla_LightGCN_best.pt                             9105.3 KB
  Gowalla_MF_best.pt                                   9105.2 KB
  Gowalla_RecDCL_best.pt                               9121.6 KB
  all_results.json                                     32.0 KB
  cross_dataset_comparison.png                         131.0 KB
  curves_Amazon_Beauty.png                             184.8 KB
  curves_Gowalla.png                                   188.6 KB
  lambda_evolution_Amazon_Beauty.png                   69.0 KB
  lambda_evolution_Gowalla.png                         64.7 KB


---
## Step 13: Download Results

Run this cell to download all output files from Colab to your local machine.


In [16]:
from google.colab import files
import os

print("Downloading all results...")
for fname in sorted(os.listdir("results_a3")):
    fpath = f"results_a3/{fname}"
    print(f"  Downloading {fname}...")
    files.download(fpath)

print("\n✓ All files downloaded!")
print("\nFiles to share with Claude:")
print("  - all_results.json")
print("  - curves_Gowalla.png")
print("  - curves_Amazon_Beauty.png")
print("  - lambda_evolution_Gowalla.png")
print("  - lambda_evolution_Amazon_Beauty.png")
print("  - cross_dataset_comparison.png")
print("  - Copy-paste the final results table from cell above")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>